In [ ]:
# [1/4] Mount Google Drive & Setup Path
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

print('Project root: /content/drive/MyDrive/code')

In [ ]:
# [2/4] Install Dependencies
!pip install -q z3-solver python-Levenshtein evaluate bert_score nltk 2>/dev/null
import nltk
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

print('Ready.')

In [ ]:
# [3/4] CONFIG — change these

MODEL = "Qwen8b"
DATASET = "val"

# Z3 timeout per candidate (ms)
Z3_TIMEOUT_MS = 10_000

print(f"Model:      {MODEL}")
print(f"Dataset:    {DATASET}")
print(f"Z3 timeout: {Z3_TIMEOUT_MS}ms")

In [ ]:
# [4/4] Compute Per-Candidate Metrics & Save
#
# Reads raw k10 candidates, computes 9 metrics per candidate vs GT,
# saves to k10 folder with checkpoint support.

import json
import time
from pathlib import Path

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

_ROOT = Path('/content/drive/MyDrive/code')
DATA_DIR = _ROOT / "data"
RESULTS_DIR = DATA_DIR / "results" / MODEL / "k10"

# ---- paths ----
_suffix = DATASET.replace("test_", "").replace("test", "")
_tag = f"{MODEL}_k10_{_suffix}" if _suffix else f"{MODEL}_k10"

k10_path = RESULTS_DIR / f"{_tag}.json"
gt_path = DATA_DIR / f"{DATASET}.json"
metrics_path = RESULTS_DIR / f"{_tag}_metrics.json"

for p, name in [(k10_path, "k10"), (gt_path, "GT")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} not found: {p}")

print("=" * 55)
print(f"  K10 METRICS COMPUTATION")
print(f"  Model:   {MODEL}")
print(f"  Dataset: {DATASET}")
print(f"  Input:   {k10_path}")
print(f"  Output:  {metrics_path}")
print("=" * 55)

# ---- load ----
with open(k10_path, encoding='utf-8') as f:
    k10_data = json.load(f)
with open(gt_path, encoding='utf-8') as f:
    gt_data = json.load(f)

gt_map = {i: item["FOL"] for i, item in enumerate(gt_data)}
n_sentences = len(k10_data)
n_cands = sum(len(e.get("candidates", [])) for e in k10_data)
print(f"\n  Sentences:  {n_sentences}")
print(f"  Candidates: {n_cands}")

# ---- resume from checkpoint ----
already_done = 0
all_results = []
if metrics_path.exists():
    with open(metrics_path, encoding='utf-8') as f:
        prev = json.load(f)
    if isinstance(prev, list) and len(prev) > 0:
        # Count entries with complete metrics on all candidates
        already_done = sum(
            1 for e in prev
            if all("metrics" in c for c in e.get("candidates", []))
        )
        if already_done > 0:
            all_results = prev
            print(f"  Resuming: {already_done}/{n_sentences} sentences already done\n")

# ---- compute ----
from src.eval.eval import compute_all_metrics

t0 = time.perf_counter()
total_processed = 0

for sent_idx in range(already_done, n_sentences):
    r = k10_data[sent_idx]
    sid = r["sentence_id"]
    gt = gt_map.get(sid, "")

    sent_entry = {
        "sentence_id": sid,
        "nl": r.get("nl", ""),
        "gt_fol": gt,
        "candidates": [],
    }

    for c in r.get("candidates", []):
        fol = c.get("fol", "") or ""
        if fol:
            metrics = compute_all_metrics(fol, gt, z3_timeout_ms=Z3_TIMEOUT_MS)
        else:
            metrics = {}
        sent_entry["candidates"].append({
            "candidate_id": c.get("candidate_id"),
            "fol": fol,
            "fol_hash": c.get("fol_hash"),
            "metrics": metrics,
        })

    all_results.append(sent_entry)
    total_processed = len(all_results)

    if total_processed % 50 == 0 or total_processed == n_sentences:
        elapsed = (time.perf_counter() - t0) / 60
        rate = total_processed / max(elapsed, 0.01)
        eta = (n_sentences - total_processed) / max(rate, 0.01)
        print(f"  [{total_processed:>4}/{n_sentences}]  "
              f"rate={rate:.0f} sent/min  elapsed={elapsed:.1f}m  ETA={eta:.1f}m")

        # checkpoint
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)

elapsed = (time.perf_counter() - t0) / 60

# ---- final save ----
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

# ---- quick stats ----
n_z3_le = 0
n_em = 0
n_total_ok = 0
for sent in all_results:
    for c in sent["candidates"]:
        m = c.get("metrics", {})
        if m:
            n_total_ok += 1
            if m.get("z3_le", 0) >= 1:
                n_z3_le += 1
            if m.get("exact_match", 0) >= 1:
                n_em += 1

print(f"\n  {'─' * 40}")
print(f"  Saved:    {metrics_path}")
print(f"  Time:     {elapsed:.1f} min")
print(f"  Valid:    {n_total_ok}/{n_cands} candidates")
print(f"  Z3 LE:    {n_z3_le}/{n_total_ok} ({100*n_z3_le/max(n_total_ok,1):.1f}%)")
print(f"  EM:       {n_em}/{n_total_ok} ({100*n_em/max(n_total_ok,1):.1f}%)")
print("=" * 55)